## はじめに（4時間パートナー版）

このノートブックでは、GlacierStyle ECサイトの非構造化データに対するCortex Search Serviceを作成します。

**前提条件:**
- part3_data_process.ipynb が実行済みであること

**作成するCortex Search Service（3つ）:**
1. FAQ用検索サービス（AI Studio GUIで作成）
2. 音声ログ要約用検索サービス
3. SNS投稿分析用（マルチインデックス）検索サービス

> **Note:** 通常版では5つのサービスを作成しますが、4時間版では代表的な3つに絞ります。

**所要時間目安:** 約30分

In [ ]:
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するウェアハウスとスキーマを設定
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 1. Cortex Search Service 概要

本ハンズオンでは、以下の3つのCortex Search Serviceを作成します。

| # | サービス名 | 対象データ | 検索カラム | 作成方法 |
|---|-----------|-----------|-----------|----------|
| 1 | FAQ検索 | gold_faq_documents | CONTENT_CHUNK | AI Studio GUI |
| 2 | 音声ログ検索 | gold_voice_logs | TRANSCRIBED_TEXT_SUMMARY | SQL |
| 3 | SNSマルチインデックス | gold_sns_mentions_with_product_master | CONTENT + PRODUCT_NAME | SQL |

**Cortex Search の特徴:**
- キーワード検索とセマンティック検索のハイブリッド
- 属性フィルター、数値ブースト、時間減衰などの高度な機能
- マルチインデックスで複数カラムの独立した検索が可能

## 2. 対象データの確認

Cortex Search Serviceを作成する前に、対象となるGold層テーブルのデータを確認します。

### 2-1. FAQドキュメントのデータ構造確認

In [ ]:
-- ============================================================================
-- FAQドキュメントのサンプルデータ確認
-- ============================================================================
SELECT 
    RELATIVE_PATH,
    LEFT(CONTENT_CHUNK, 200) AS content_preview
FROM GOLD_FAQ_DOCUMENTS
LIMIT 5;

### 2-3. 音声ログのデータ構造確認

In [ ]:
-- ============================================================================
-- 音声ログのサンプルデータ確認
-- ============================================================================
SELECT 
    CALL_ID,
    CATEGORY,
    INQUIRY_CATEGORY,
    SENTIMENT,
    LEFT(TRANSCRIBED_TEXT_SUMMARY, 100) AS summary_preview
FROM GOLD_VOICE_LOGS
LIMIT 5;

### 2-4. SNS投稿のデータ構造確認

In [ ]:
-- ============================================================================
-- SNS投稿のサンプルデータ確認
-- SENTIMENTカラム: positive / neutral / negative / mixed
-- POSTED_ATカラム: VARCHAR型（形式: YYYY/MM/DD HH24:MI:SS）
-- ============================================================================
SELECT 
    POST_ID,
    PLATFORM,
    USERNAME,
    DISPLAY_NAME,
    EXTRACTED_PRODUCT_NAME,
    SENTIMENT,
    LIKES,
    RETWEETS,
    POSTED_AT,
    LEFT(CONTENT, 100) AS content_preview
FROM GOLD_SNS_MENTIONS_ANALYZED
LIMIT 5;

## 3. Cortex Search Serviceの作成

各データソースに対してCortex Search Serviceを作成します。

### 3-1. FAQ用 Cortex Search Service

**このサービスはAI StudioのGUI画面で作成します。**

以下の手順でAI Studioから作成してください：

1. Snowsight → **AI と ML** → **検索** をクリック
2. 「**作成**」ボタンをクリック
4. 以下の設定を入力：
   - データベース: `GLACIERSTYLE_DB`
   - スキーマ: `EC_ANALYTICS_SCHEMA`
   - サービス名: `SEARCH_FAQ`
   - インデックス対象テーブル: `GOLD_FAQ_DOCUMENTS`
   - 検索列: `CONTENT_CHUNK`
   - 属性列: なにも選ばない
   - サービスに含む列: すべて選択
   - ターゲットラグ: `1 day`
   - 埋め込みモデル: `snowflake-arctic-embed-l-v2.0`
   - ウェアハウス: `GLACIERSTYLE_WH`
5. 「**作成**」をクリック

---

> **💡 AI Studioで作成しなかった場合**  
> 以下のSQLのコメントアウトを解除して実行してください。

In [ ]:
-- ============================================================================
-- FAQ用 Cortex Search Service の作成
-- ※このセクションはAI StudioでGUI作成した場合はスキップしてください
-- ※AI Studioで作成しなかった場合は、以下のコメントアウトを解除して実行してください
-- ============================================================================

/*
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_FAQ
  ON CONTENT_CHUNK
  WAREHOUSE = GLACIERSTYLE_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT 
        RELATIVE_PATH,
        CONTENT_CHUNK
    FROM GOLD_FAQ_DOCUMENTS
    WHERE CONTENT_CHUNK IS NOT NULL
      AND LENGTH(TRIM(CONTENT_CHUNK)) > 0
  );
*/

### 3-2. 音声ログ要約用 Cortex Search Service

コールセンターの通話要約を検索するサービスを作成します。

**設定項目:**
- サービス名: SEARCH_VOICE_LOGS
- 対象テーブル: GOLD_VOICE_LOGS
- 検索列: TRANSCRIBED_TEXT_SUMMARY
- 属性列: CALL_ID, CATEGORY, INQUIRY_CATEGORY, SENTIMENT
- ターゲットラグ: 1日
- 埋め込みモデル: snowflake-arctic-embed-l-v2.0

> **属性列（ATTRIBUTES）とは？**  
> 検索結果をフィルタリングするためのカラムです。  
> 例: 感情が「negative」の問い合わせのみを検索

> **SENTIMENTカラムの値:**  
> positive / neutral / negative / mixed

In [ ]:
-- ============================================================================
-- 音声ログ要約用 Cortex Search Service の作成
-- ============================================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_VOICE_LOGS
  ON TRANSCRIBED_TEXT_SUMMARY
  ATTRIBUTES CALL_ID, CATEGORY, INQUIRY_CATEGORY, SENTIMENT
  WAREHOUSE = GLACIERSTYLE_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT 
        CALL_ID,
        SCENARIO_ID,
        CATEGORY,
        INQUIRY_CATEGORY,
        SENTIMENT,
        AGENT_ID,
        CALL_DURATION_SEC,
        CALL_START_TIME,
        TRANSCRIBED_TEXT_SUMMARY
    FROM GOLD_VOICE_LOGS
    WHERE TRANSCRIBED_TEXT_SUMMARY IS NOT NULL
      AND LENGTH(TRIM(TRANSCRIBED_TEXT_SUMMARY)) > 0
  );

### 3-3. SNS投稿分析用 Cortex Search Service

SNS投稿データ（GOLD_SNS_MENTIONS_ANALYZED）をセマンティック検索できるようにします。

- **検索対象列**: CONTENT（投稿内容）
- **属性列**: PLATFORM, POST_CATEGORY, SENTIMENT（フィルター検索用）
- **数値列**: LIKES, RETWEETS（Numeric Boost用）
- **時刻列**: POSTED_AT（Time Decay用）

> **活用例**  
> 「GlacierStyle インテリア」で検索 → 関連するSNS投稿を意味的に検索  
> フィルターで「Twitterのポジティブ投稿のみ」に絞り込み  
> Numeric Boostで「いいね数が多い投稿」を優先  
> Time Decayで「最新の投稿」を優先

In [ ]:
-- ============================================================================
-- SNS投稿分析用 Cortex Search Service の作成
-- POSTED_ATをTIMESTAMP型にキャストして、Time Decay機能を利用可能にする
-- 元データ形式: 'YYYY/MM/DD HH24:MI:SS'
-- ============================================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_SNS_MENTIONS
  ON CONTENT
  ATTRIBUTES PLATFORM, POST_CATEGORY, SENTIMENT
  WAREHOUSE = GLACIERSTYLE_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT 
        POST_ID,
        PLATFORM,
        POST_TYPE,
        USERNAME,
        DISPLAY_NAME,
        CONTENT,
        TO_TIMESTAMP_NTZ(POSTED_AT) AS POSTED_AT,
        LIKES,
        RETWEETS,
        REPLIES,
        POST_CATEGORY,
        SENTIMENT,
        EXTRACTED_CATEGORY,
        EXTRACTED_PRODUCT_NAME
    FROM GOLD_SNS_MENTIONS_ANALYZED
    WHERE CONTENT IS NOT NULL
      AND LENGTH(TRIM(CONTENT)) > 0
  );

### 3-4. マルチインデックス用 Cortex Search Service

**マルチインデックス機能**を使うと、複数のカラムに対してそれぞれ異なる特性を持つインデックスを作成し、検索時に組み合わせて利用できます。

- **TEXT INDEXES**: キーワード（字句）検索用のインデックス。完全一致やキーワードマッチが重要なフィールドに適している。
- **VECTOR INDEXES**: ベクトル（意味）検索用のインデックス。意味的な類似性での検索が重要なフィールドに適している。

**設定項目:**
- サービス名: SEARCH_SNS_MULTI_INDEX
- TEXT INDEXES: USERNAME, DISPLAY_NAME, EXTRACTED_PRODUCT_NAME（キーワード検索用）
- VECTOR INDEXES: CONTENT（意味検索用）
- 属性列: PLATFORM, SENTIMENT

> **活用例**  
> 「@user123」で検索 → ユーザー名にマッチ（TEXT INDEX）  
> 「おしゃれなライト」で検索 → 意味的に関連する投稿がマッチ（VECTOR INDEX）

> **注意**: 現時点ではSnowsight GUIからはマルチインデックスのCortex Search Serviceを作成できません。SQLで作成する必要があります。

In [ ]:
-- ============================================================================
-- マルチインデックス用 Cortex Search Service の作成
-- TEXT INDEXES: キーワード（字句）検索用
-- VECTOR INDEXES: ベクトル（意味）検索用
-- ============================================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE SEARCH_SNS_MULTI_INDEX
    TEXT INDEXES USERNAME, DISPLAY_NAME, EXTRACTED_PRODUCT_NAME
    VECTOR INDEXES CONTENT (model='snowflake-arctic-embed-l-v2.0')
    ATTRIBUTES PLATFORM, SENTIMENT
    WAREHOUSE = GLACIERSTYLE_WH
    TARGET_LAG = '1 day'
    AS (
        SELECT 
            POST_ID,
            PLATFORM,
            USERNAME,
            DISPLAY_NAME,
            CONTENT,
            TO_TIMESTAMP_NTZ(POSTED_AT) AS POSTED_AT,
            LIKES,
            RETWEETS,
            SENTIMENT,
            EXTRACTED_CATEGORY,
            EXTRACTED_PRODUCT_NAME
        FROM GOLD_SNS_MENTIONS_ANALYZED
        WHERE CONTENT IS NOT NULL
    );

## 4. Cortex Search Service の確認

作成したCortex Search Serviceの一覧と状態を確認します。

In [ ]:
-- ============================================================================
-- Cortex Search Serviceの一覧確認
-- ============================================================================
SHOW CORTEX SEARCH SERVICES IN SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 5. 検索テスト（基本検索）

作成したCortex Search Serviceを使用して、基本的な検索を行います。

**SEARCH_PREVIEW関数の結果をフラット化して見やすく表示します。**

> **注意**: Cortex Searchの検索結果は、すでに最適な順序でランキングされて返されます。  
> numeric_boostsやtime_decaysの効果もこの順序に反映されています。

### 5-1. FAQの検索テスト

In [ ]:
-- ============================================================================
-- FAQ検索テスト: 返品に関するFAQ
-- ============================================================================
SELECT 
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:RELATIVE_PATH::STRING AS relative_path,
    LEFT(f.value:CONTENT_CHUNK::STRING, 300) AS content_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_FAQ',
        '{
            "query": "返品の条件を教えてください",
            "columns": ["RELATIVE_PATH", "CONTENT_CHUNK"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 5-3. 音声ログ要約の検索テスト（基本検索）

In [ ]:
-- ============================================================================
-- 音声ログ検索テスト: 配送遅延に関する問い合わせ
-- ============================================================================
SELECT 
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:CALL_ID::STRING AS call_id,
    f.value:CATEGORY::STRING AS category,
    f.value:SENTIMENT::STRING AS sentiment,
    LEFT(f.value:TRANSCRIBED_TEXT_SUMMARY::STRING, 200) AS summary_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_VOICE_LOGS',
        '{
            "query": "配送が遅れている",
            "columns": ["CALL_ID", "CATEGORY", "SENTIMENT", "TRANSCRIBED_TEXT_SUMMARY"],
            "limit": 3
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## 6. 高度な検索機能

Cortex Searchでは、基本的なセマンティック検索に加えて、以下の高度な機能を利用できます。

**1. 属性フィルター**: 特定の条件でデータを絞り込み  
**2. Numeric Boosts**: 数値カラム（いいね数など）でスコアをブースト  
**3. Time Decays**: タイムスタンプによる減衰（最新を優先）  
**4. マルチインデックス検索**: TEXT + VECTOR インデックスを横断した検索

### 6-1. 属性フィルター検索

**ATTRIBUTES** で指定したカラムを使って、検索結果をフィルタリングできます。

例: ネガティブな感情の問い合わせのみを検索

In [ ]:
-- ============================================================================
-- 属性フィルター検索: ネガティブな問い合わせのみ
-- SENTIMENT列: positive / neutral / negative / mixed
-- ============================================================================
SELECT 
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:CALL_ID::STRING AS call_id,
    f.value:CATEGORY::STRING AS category,
    f.value:INQUIRY_CATEGORY::STRING AS inquiry_category,
    f.value:SENTIMENT::STRING AS sentiment,
    LEFT(f.value:TRANSCRIBED_TEXT_SUMMARY::STRING, 150) AS summary_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_VOICE_LOGS',
        '{
            "query": "商品の不具合",
            "columns": ["CALL_ID", "CATEGORY", "INQUIRY_CATEGORY", "SENTIMENT", "TRANSCRIBED_TEXT_SUMMARY"],
            "filter": {"@eq": {"SENTIMENT": "negative"}},
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- 属性フィルター検索: Twitterのポジティブな投稿のみ
-- ============================================================================
SELECT 
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:POST_ID::STRING AS post_id,
    f.value:PLATFORM::STRING AS platform,
    f.value:SENTIMENT::STRING AS sentiment,
    f.value:LIKES::INTEGER AS likes,
    LEFT(f.value:CONTENT::STRING, 100) AS content_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MENTIONS',
        '{
            "query": "おすすめ 購入",
            "columns": ["POST_ID", "PLATFORM", "SENTIMENT", "LIKES", "CONTENT"],
            "filter": {
                "@and": [
                    {"@eq": {"PLATFORM": "twitter"}},
                    {"@eq": {"SENTIMENT": "positive"}}
                ]
            },
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

### 6-2. Numeric Boosts（エンゲージメントによるブースト）

**numeric_boosts** を使うと、数値カラムの値に基づいて検索スコアをブーストできます。

SNS投稿の場合：
- **LIKES（いいね数）** が多い投稿を優先
- **RETWEETS（リツイート数）** が多い投稿を優先

これにより、エンゲージメントの高い（＝信頼性が高い）投稿を上位に表示できます。

---

**まずはブーストなしで検索し、その後ブーストありと比較します。**

In [ ]:
-- ============================================================================
-- 【比較用】ブーストなし: セマンティック検索のみ
-- ============================================================================
SELECT 
    'ブーストなし' AS search_type,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:POST_ID::STRING AS post_id,
    f.value:PLATFORM::STRING AS platform,
    f.value:LIKES::INTEGER AS likes,
    f.value:RETWEETS::INTEGER AS retweets,
    LEFT(f.value:CONTENT::STRING, 80) AS content_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MENTIONS',
        '{
            "query": "GlacierStyle インテリア",
            "columns": ["POST_ID", "PLATFORM", "LIKES", "RETWEETS", "CONTENT"],
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- Numeric Boosts: いいね数・リツイート数でスコアを強くブースト
-- LIKES weight=50, RETWEETS weight=30 → 強めのブースト設定
-- ============================================================================
SELECT 
    'ブーストあり' AS search_type,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:POST_ID::STRING AS post_id,
    f.value:PLATFORM::STRING AS platform,
    f.value:LIKES::INTEGER AS likes,
    f.value:RETWEETS::INTEGER AS retweets,
    LEFT(f.value:CONTENT::STRING, 80) AS content_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MENTIONS',
        '{
            "query": "GlacierStyle インテリア",
            "columns": ["POST_ID", "PLATFORM", "LIKES", "RETWEETS", "CONTENT"],
            "scoring_config": {
                "functions": {
                    "numeric_boosts": [
                        {"column": "LIKES", "weight": 50},
                        {"column": "RETWEETS", "weight": 30}
                    ]
                }
            },
            "limit": 5
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

> **💡 比較のポイント**  
> - ブーストなしでは、意味的に近い投稿が上位に来る
> - ブーストありでは、LIKES/RETWEETSが多い投稿が上位に来やすい
> - エンゲージメントの高い投稿 ≒ 信頼性が高い・有益な情報

### 6-3. Time Decays（時間による減衰）

**time_decays** を使うと、タイムスタンプが新しいデータほど高スコアになります。

パラメータ:
- **column**: タイムスタンプカラム（TIMESTAMP型である必要あり）
- **weight**: 複数のtime_decaysを設定した場合の相対的な重要度（単一の場合は影響なし）
- **limit_hours**: この時間より古いデータはブーストがほぼゼロに（今回は720時間＝30日）
- **now**: 減衰を計算するための基準時刻（ISO-8601形式）※ハンズオンでは固定値を指定

これにより、最新の顧客の声を優先的に取得できます。

> **注意**: SNSデータは2024年12月のデータです。  
> `now`パラメータに2025年1月1日を指定し、`limit_hours`を720時間（30日）に設定することで、  
> 12月初旬のデータは減衰効果が大きく、12月中旬以降のデータは上位に来やすくなります。

---

**まずは減衰なしで検索し、その後減衰ありと比較します。**

In [ ]:
-- ============================================================================
-- 【比較用】減衰なし: セマンティック検索のみ
-- ============================================================================
SELECT 
    '減衰なし' AS search_type,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:POST_ID::STRING AS post_id,
    f.value:PLATFORM::STRING AS platform,
    TO_VARCHAR(f.value:POSTED_AT::TIMESTAMP_NTZ, 'YYYY-MM-DD HH24:MI') AS posted_at,
    f.value:LIKES::INTEGER AS likes,
    LEFT(f.value:CONTENT::STRING, 80) AS content_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MENTIONS',
        '{
            "query": "GlacierStyle 購入",
            "columns": ["POST_ID", "PLATFORM", "POSTED_AT", "LIKES", "CONTENT"],
            "limit": 10
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

In [ ]:
-- ============================================================================
-- Time Decays: 最新の投稿を優先
-- limit_hours=720 → 30日以上前の投稿はブーストがほぼゼロ
-- now=2025-01-01T00:00:00.000+09:00 → 基準時刻を固定（データは2024年12月）
-- ============================================================================
SELECT 
    '減衰あり' AS search_type,
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:POST_ID::STRING AS post_id,
    f.value:PLATFORM::STRING AS platform,
    TO_VARCHAR(f.value:POSTED_AT::TIMESTAMP_NTZ, 'YYYY-MM-DD HH24:MI') AS posted_at,
    f.value:LIKES::INTEGER AS likes,
    LEFT(f.value:CONTENT::STRING, 80) AS content_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MENTIONS',
        '{
            "query": "GlacierStyle 購入",
            "columns": ["POST_ID", "PLATFORM", "POSTED_AT", "LIKES", "CONTENT"],
            "scoring_config": {
                "functions": {
                    "time_decays": [
                        {
                            "column": "POSTED_AT",
                            "weight": 5,
                            "limit_hours": 720,
                            "now": "2025-01-01T00:00:00.000+09:00"
                        }
                    ]
                }
            },
            "limit": 10
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

> **💡 比較のポイント**  
> - 減衰なしでは、投稿日時に関係なく意味的に近い投稿が上位
> - 減衰ありでは、基準時刻（2025年1月1日）に近い12月中旬以降の投稿が上位に来やすい
> - 12月初旬の投稿は、limit_hours=720時間（30日）の境界を超えるため減衰効果が大きい
> - トレンド分析や最新の顧客の声を把握したい場合に有効

### 6-4. マルチインデックス検索

**マルチインデックス検索**では、`multi_index_query` パラメータを使用して、各インデックスに対して別々のクエリを投げることができます。

- **TEXT INDEX**: キーワードマッチさせたい文字列を指定
- **VECTOR INDEX**: 意味検索させたいクエリを指定

さらに、`scoring_config` の `weights` パラメータで **キーワード検索 (texts) とベクトル検索 (vectors) の重みバランス** を調整できます。

---

**ケース1: TEXT検索（キーワード検索）を重視**

In [ ]:
-- ============================================================================
-- マルチインデックス検索: TEXT検索（キーワード）を重視
-- weights: texts=5, vectors=1 → キーワードマッチを優先
-- ============================================================================
SELECT
    'TEXT重視' AS search_type,
    value:POST_ID::STRING AS post_id,
    value:USERNAME::STRING AS username,
    value:EXTRACTED_PRODUCT_NAME::STRING AS product_name,
    LEFT(value:CONTENT::STRING, 100) AS content_preview
FROM TABLE(FLATTEN(PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MULTI_INDEX',
        '{
            "multi_index_query": {
                "EXTRACTED_PRODUCT_NAME": [{"text": "デスクライト"}],
                "CONTENT": [{"text": "おしゃれなインテリア"}]
            },
            "scoring_config": {
                "weights": {
                    "texts": 5,
                    "vectors": 1,
                    "reranker": 1
                }
            },
            "columns": ["POST_ID", "USERNAME", "EXTRACTED_PRODUCT_NAME", "CONTENT"],
            "limit": 5
        }'
    )
)['results']));

**ケース2: VECTOR検索（意味検索）を重視**

In [ ]:
-- ============================================================================
-- マルチインデックス検索: VECTOR検索（意味検索）を重視
-- weights: texts=1, vectors=5 → 意味的なマッチを優先
-- ============================================================================
SELECT
    'VECTOR重視' AS search_type,
    value:POST_ID::STRING AS post_id,
    value:USERNAME::STRING AS username,
    value:EXTRACTED_PRODUCT_NAME::STRING AS product_name,
    LEFT(value:CONTENT::STRING, 100) AS content_preview
FROM TABLE(FLATTEN(PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MULTI_INDEX',
        '{
            "multi_index_query": {
                "EXTRACTED_PRODUCT_NAME": [{"text": "デスクライト"}],
                "CONTENT": [{"text": "おしゃれなインテリア"}]
            },
            "scoring_config": {
                "weights": {
                    "texts": 1,
                    "vectors": 5,
                    "reranker": 1
                }
            },
            "columns": ["POST_ID", "USERNAME", "EXTRACTED_PRODUCT_NAME", "CONTENT"],
            "limit": 5
        }'
    )
)['results']));

> **💡 マルチインデックス検索のポイント**  
> - **各インデックスに別々のクエリを投げられる**: TEXT INDEXにはキーワード、VECTOR INDEXには意味クエリ
> - **weights で検索タイプの重みを調整**: texts（キーワード）vs vectors（意味）
> - TEXT検索は完全一致/部分一致、VECTOR検索は意味的な類似性で検索

### 6-5. 組み合わせ検索（フィルター + ブースト + 減衰）

これらの機能を組み合わせることで、より精度の高い検索が可能です。

例: **Twitterのポジティブな投稿**から、**エンゲージメントが高く**、**最新の**ものを検索

In [ ]:
-- ============================================================================
-- 組み合わせ検索: フィルター + ブースト + 減衰
-- Twitterのポジティブな投稿から、エンゲージメントが高く最新のものを検索
-- limit_hours=720, now=2025-01-01T00:00:00.000+09:00 → 基準時刻を固定
-- ============================================================================
SELECT 
    ROUND(f.value:"@scores":cosine_similarity::FLOAT, 3) AS similarity,
    f.value:POST_ID::STRING AS post_id,
    f.value:PLATFORM::STRING AS platform,
    f.value:SENTIMENT::STRING AS sentiment,
    TO_VARCHAR(f.value:POSTED_AT::TIMESTAMP_NTZ, 'YYYY-MM-DD HH24:MI') AS posted_at,
    f.value:LIKES::INTEGER AS likes,
    f.value:RETWEETS::INTEGER AS retweets,
    LEFT(f.value:CONTENT::STRING, 80) AS content_preview
FROM (
    SELECT SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SEARCH_SNS_MENTIONS',
        '{
            "query": "おしゃれ インテリア",
            "columns": ["POST_ID", "PLATFORM", "SENTIMENT", "POSTED_AT", "LIKES", "RETWEETS", "CONTENT"],
            "filter": {
                "@and": [
                    {"@eq": {"PLATFORM": "twitter"}},
                    {"@eq": {"SENTIMENT": "positive"}}
                ]
            },
            "scoring_config": {
                "functions": {
                    "numeric_boosts": [
                        {"column": "LIKES", "weight": 50},
                        {"column": "RETWEETS", "weight": 30}
                    ],
                    "time_decays": [
                        {
                            "column": "POSTED_AT",
                            "weight": 5,
                            "limit_hours": 720,
                            "now": "2025-01-01T00:00:00.000+09:00"
                        }
                    ]
                }
            },
            "limit": 10
        }'
    ) AS result_json
),
LATERAL FLATTEN(input => PARSE_JSON(result_json):results) AS f;

## まとめ

このノートブックでは、3つのCortex Search Serviceを作成しました。

### 作成したサービス

| サービス名 | 対象 | 検索タイプ |
|-----------|------|----------|
| FAQ検索 | gold_faq_documents | 基本検索 |
| 音声ログ検索 | gold_voice_logs | フィルター + 属性検索 |
| SNSマルチインデックス | gold_sns_mentions_with_product_master | マルチインデックス |

### 学んだ高度な機能

- **属性フィルター**: カラム値で検索結果を絞り込み
- **Numeric Boosts**: エンゲージメント（いいね数等）で重み付け
- **Time Decays**: 新しいデータほど高スコア
- **マルチインデックス**: 複数カラムで独立した検索

### 次のステップ

- **Part 6**: Cortex Agentの作成
  - 上記3つのCortex Searchツール + Semantic View + Web検索を統合したエージェントを構築